# LeetCode 2196 — Create Binary Tree From Descriptions bằng Python

Link đề: https://leetcode.com/problems/create-binary-tree-from-descriptions/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Tree, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def createBinaryTree(self, descriptions: List[List[int]]) -> Optional[TreeNode]:
        ...
```

Mỗi mô tả `[parent, child, isLeft]` cho biết một cạnh cha–con. Dựng cây và trả về **gốc** (đỉnh duy nhất **không** là con của ai). Notebook tự cài `TreeNode` + hàm tuần tự hóa kiểu LeetCode (BFS, cắt `null` cuối), rồi đi từ **tìm gốc vét cạn `O(n²)`** → **từ điển nút + tập con** → **đếm bậc vào** → **mẹo tổng giá trị (gốc = tổng cha − tổng con)** → **union-find "ai là cha của ai" / đi ngược lên từ một nút bất kỳ**.

## 1. Mục tiêu học

1. Dùng **từ điển `giá trị → TreeNode`** để mỗi giá trị chỉ có **một** nút (tạo lười khi gặp lần đầu).
2. Xác định gốc: đỉnh xuất hiện làm **cha** nhưng không bao giờ làm **con**.
3. Tự viết `TreeNode`, `serialize` (BFS kiểu LeetCode) và `deserialize` để kiểm thử.
4. Viết **validator**: duyệt cây kết quả, sinh lại tập mô tả, so với đầu vào (thứ tự bất kỳ).
5. Tránh lỗi: tạo nút trùng, lấy `descriptions[0][0]` làm gốc, đảo trái/phải, duyệt đệ quy quá sâu.

## 2. Đề bài & ví dụ

`descriptions[i] = [parentᵢ, childᵢ, isLeftᵢ]`: `childᵢ` là con của `parentᵢ` trong một cây nhị phân có giá trị **phân biệt**; `isLeftᵢ = 1` ⇒ con **trái**, `0` ⇒ con **phải**. Dựng cây và trả về gốc. Dữ liệu đảm bảo cây hợp lệ.

**Ví dụ 1:** `[[20,15,1],[20,17,0],[50,20,1],[50,80,0],[80,19,1]]` → `[50,20,80,15,17,19]`.

**Ví dụ 2:** `[[1,2,1],[2,3,0],[3,4,1]]` → `[1,2,null,null,3,4]`.

**Ràng buộc:** `1 ≤ len(descriptions) ≤ 10⁴`, `1 ≤ parent, child ≤ 10⁵`, `isLeft ∈ {0, 1}`.

In [1]:
import random
import sys
import time
from collections import deque
from typing import List, Optional

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def serialize(root):
    """Kiểu LeetCode: BFS, None cho chỗ trống, cắt các None ở cuối."""
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None); continue
        out.append(node.val)
        q.append(node.left); q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

def deserialize(data):
    if not data:
        return None
    it = iter(data)
    root = TreeNode(next(it)); q = deque([root])
    for v in it:
        node = q[0]
        if not hasattr(node, "_filled"):
            node._filled = 0
        child = None if v is None else TreeNode(v)
        if node._filled == 0:
            node.left = child; node._filled = 1
        else:
            node.right = child; q.popleft()
        if child:
            q.append(child)
    return root

examples = [([[20, 15, 1], [20, 17, 0], [50, 20, 1], [50, 80, 0], [80, 19, 1]], [50, 20, 80, 15, 17, 19]),
            ([[1, 2, 1], [2, 3, 0], [3, 4, 1]], [1, 2, None, None, 3, 4])]
for _, exp in examples:
    print("kỳ vọng", exp, "-> dựng rồi tuần tự hóa lại:", serialize(deserialize(exp)))

kỳ vọng [50, 20, 80, 15, 17, 19] -> dựng rồi tuần tự hóa lại: [50, 20, 80, 15, 17, 19]
kỳ vọng [1, 2, None, None, 3, 4] -> dựng rồi tuần tự hóa lại: [1, 2, None, None, 3, 4]


## 3. Kiến thức nền

### 3.1 Từ điển tạo nút lười

In [2]:
nodes = {}
def get(v):
    if v not in nodes:
        nodes[v] = TreeNode(v)
    return nodes[v]

a1 = get(20); a2 = get(20)
print("cùng một nút?", a1 is a2, "| số nút:", len(nodes))

cùng một nút? True | số nút: 1


### 3.2 Gốc = cha mà không là con

In [3]:
desc = examples[0][0]
parents = {p for p, _, _ in desc}; children = {c for _, c, _ in desc}
print("cha:", sorted(parents), "| con:", sorted(children), "| cha − con =", parents - children)

cha: [20, 50, 80] | con: [15, 17, 19, 20, 80] | cha − con = {50}


### 3.3 In cây dạng cây thư mục (DFS lặp)

In [4]:
def draw(root):
    lines, stack = [], [(root, 0, "gốc")]
    while stack:
        node, d, tag = stack.pop()
        if node is None:
            continue
        lines.append("    " + "  " * d + f"{tag}: {node.val}")
        stack.append((node.right, d + 1, "P"))
        stack.append((node.left, d + 1, "T"))
    print("\n".join(lines))

draw(deserialize(examples[0][1]))

    gốc: 50
      T: 20
        T: 15
        P: 17
      P: 80
        T: 19


### 3.4 Validator: sinh lại mô tả từ cây

In [5]:
def descriptions_of(root):
    out, stack = set(), [root]
    while stack:
        node = stack.pop()
        if node.left:
            out.add((node.val, node.left.val, 1)); stack.append(node.left)
        if node.right:
            out.add((node.val, node.right.val, 0)); stack.append(node.right)
    return out

print(sorted(descriptions_of(deserialize(examples[0][1]))) == sorted(map(tuple, examples[0][0])))

True


## 4. Ý tưởng nền tảng / trực giác

Mỗi mô tả nói "nối `child` vào phía trái/phải của `parent`". Thứ tự mô tả **tùy ý** — có thể gặp `20` làm cha trước khi biết `20` là con của `50` (ví dụ 1). Vì vậy cần một chỗ lưu "nút của giá trị `v`" để mọi mô tả cùng trỏ vào **một** nút: từ điển.

Gốc là đỉnh **không có cha**. Trong cây, mọi đỉnh trừ gốc xuất hiện đúng một lần ở cột `child`. Ví dụ 1: con `= {15, 17, 20, 80, 19}`, cha `= {20, 50, 80}` ⇒ cha − con `= {50}` ⇒ gốc 50 ✓.

Mẹo số học: mọi đỉnh trừ gốc xuất hiện **đúng một lần** làm con ⇒ `gốc = Σ(mọi giá trị phân biệt) − Σ(con)`.

## 5. Cách 1 — Brute Force / baseline: tìm gốc bằng cách kiểm từng cha với mọi mô tả

Dựng nút bằng từ điển, rồi với mỗi cha `p` kiểm `p` có xuất hiện ở cột con của mô tả nào không — `O(n²)`.

In [6]:
def build_brute(descriptions):
    nodes = {}
    def get(v):
        if v not in nodes:
            nodes[v] = TreeNode(v)
        return nodes[v]
    for p, c, left in descriptions:
        if left:
            get(p).left = get(c)
        else:
            get(p).right = get(c)
    for p, _, _ in descriptions:
        if all(c != p for _, c, _ in descriptions):        # O(n) mỗi lần kiểm
            return nodes[p]
    return None

for desc_, exp in examples:
    print(serialize(build_brute(desc_)), "(kỳ vọng", exp, ")")

[50, 20, 80, 15, 17, 19] (kỳ vọng [50, 20, 80, 15, 17, 19] )
[1, 2, None, None, 3, 4] (kỳ vọng [1, 2, None, None, 3, 4] )


**Độ phức tạp:** `O(n²)` để tìm gốc — `10⁸` với `n = 10⁴`: chậm. **Nút thắt:** kiểm "là con" bằng duyệt danh sách ⇒ dùng **tập**.

## 6. Ý tưởng thuật toán / bước đột phá

**Một lượt duyệt:**
- lấy (hoặc tạo) nút `p` và `c` trong từ điển;
- **nếu** `isLeft == 1` **thì** `p.left = c`, **ngược lại** `p.right = c`;
- ghi `c` vào tập `children`.

**Gốc:** giá trị (trong các khóa của từ điển) không thuộc `children`. Duy nhất vì cây hợp lệ.

**Bất biến:** sau khi xử lý `i` mô tả, từ điển chứa đúng một nút cho mỗi giá trị đã gặp, và mọi cạnh trong `i` mô tả đầu đã được nối.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Từ điển nút + tập con

In [7]:
def build_dict_set(descriptions):
    nodes, children = {}, set()
    for p, c, left in descriptions:
        if p not in nodes: nodes[p] = TreeNode(p)
        if c not in nodes: nodes[c] = TreeNode(c)
        if left:
            nodes[p].left = nodes[c]
        else:
            nodes[p].right = nodes[c]
        children.add(c)
    for v in nodes:
        if v not in children:
            return nodes[v]

for desc_, exp in examples:
    print(serialize(build_dict_set(desc_)), "(kỳ vọng", exp, ")")

[50, 20, 80, 15, 17, 19] (kỳ vọng [50, 20, 80, 15, 17, 19] )
[1, 2, None, None, 3, 4] (kỳ vọng [1, 2, None, None, 3, 4] )


**Độ phức tạp:** `O(n)` trung bình, bộ nhớ `O(n)`.

### 7.2 Cách 3 — Đếm bậc vào (in-degree)

In [8]:
def build_indegree(descriptions):
    nodes, indeg = {}, {}
    for p, c, left in descriptions:
        for v in (p, c):
            if v not in nodes:
                nodes[v] = TreeNode(v); indeg[v] = 0
        setattr(nodes[p], "left" if left else "right", nodes[c])
        indeg[c] += 1
    root_vals = [v for v, d in indeg.items() if d == 0]
    return nodes[root_vals[0]]

for desc_, exp in examples:
    print(serialize(build_indegree(desc_)), "(kỳ vọng", exp, ")")

[50, 20, 80, 15, 17, 19] (kỳ vọng [50, 20, 80, 15, 17, 19] )
[1, 2, None, None, 3, 4] (kỳ vọng [1, 2, None, None, 3, 4] )


**Ưu điểm:** cùng `O(n)`, và nếu dữ liệu **không** hợp lệ (nhiều gốc, đỉnh có 2 cha) ta phát hiện được ngay qua `indeg` — hữu ích cho bài kiểm tra tính hợp lệ (biến thể).

### 7.3 Cách 4 — Mẹo tổng: gốc = Σ giá trị phân biệt − Σ con

In [9]:
def build_sum_trick(descriptions):
    nodes = {}
    child_sum = 0
    for p, c, left in descriptions:
        if p not in nodes: nodes[p] = TreeNode(p)
        if c not in nodes: nodes[c] = TreeNode(c)
        if left: nodes[p].left = nodes[c]
        else: nodes[p].right = nodes[c]
        child_sum += c                               # mỗi đỉnh trừ gốc là con đúng 1 lần
    return nodes[sum(nodes) - child_sum]

for desc_, exp in examples:
    print(serialize(build_sum_trick(desc_)), "(kỳ vọng", exp, ")")

[50, 20, 80, 15, 17, 19] (kỳ vọng [50, 20, 80, 15, 17, 19] )
[1, 2, None, None, 3, 4] (kỳ vọng [1, 2, None, None, 3, 4] )


**Vì sao đúng?** Mỗi đỉnh khác gốc có **đúng một** cha ⇒ xuất hiện đúng một lần ở cột con; gốc không xuất hiện. `Σ_{v ∈ V} v − Σ_{con} c = gốc`. Chỉ cần thêm `O(1)` bộ nhớ ngoài từ điển (không cần tập con). Cũng dùng được với XOR.

### 7.4 Cách 5 — Đi ngược lên bằng bản đồ "cha của"

Lưu `parent_of[c] = p`. Từ **bất kỳ** đỉnh nào, đi theo `parent_of` cho tới khi không còn cha ⇒ gốc. Độ dài đường đi ≤ chiều cao cây (≤ `n`).

In [10]:
def build_climb(descriptions):
    nodes, parent_of = {}, {}
    for p, c, left in descriptions:
        if p not in nodes: nodes[p] = TreeNode(p)
        if c not in nodes: nodes[c] = TreeNode(c)
        if left: nodes[p].left = nodes[c]
        else: nodes[p].right = nodes[c]
        parent_of[c] = p
    v = descriptions[0][0]
    while v in parent_of:
        v = parent_of[v]
    return nodes[v]

for desc_, exp in examples:
    print(serialize(build_climb(desc_)), "(kỳ vọng", exp, ")")

[50, 20, 80, 15, 17, 19] (kỳ vọng [50, 20, 80, 15, 17, 19] )
[1, 2, None, None, 3, 4] (kỳ vọng [1, 2, None, None, 3, 4] )


**Độ phức tạp:** `O(n + h)`. Ý tưởng "đi theo con trỏ cha tới gốc" là nền của union-find (`find`).

## 8. TRACE từng bước

### 8.1 Trace cách 2 trên ví dụ 1

In [11]:
def trace_dict_set(descriptions):
    nodes, children = {}, set()
    for p, c, left in descriptions:
        created = [v for v in (p, c) if v not in nodes]
        for v in created: nodes[v] = TreeNode(v)
        setattr(nodes[p], "left" if left else "right", nodes[c])
        children.add(c)
        print(f"    [{p}, {c}, {left}]: tạo {created if created else 'không'}, nối {c} vào {'TRÁI' if left else 'PHẢI'} của {p}; con = {sorted(children)}")
    roots = [v for v in nodes if v not in children]
    print("    gốc (không là con):", roots)
    return nodes[roots[0]]

r = trace_dict_set(examples[0][0])
draw(r)
print("  =>", serialize(r))

    [20, 15, 1]: tạo [20, 15], nối 15 vào TRÁI của 20; con = [15]
    [20, 17, 0]: tạo [17], nối 17 vào PHẢI của 20; con = [15, 17]
    [50, 20, 1]: tạo [50], nối 20 vào TRÁI của 50; con = [15, 17, 20]
    [50, 80, 0]: tạo [80], nối 80 vào PHẢI của 50; con = [15, 17, 20, 80]
    [80, 19, 1]: tạo [19], nối 19 vào TRÁI của 80; con = [15, 17, 19, 20, 80]
    gốc (không là con): [50]
    gốc: 50
      T: 20
        T: 15
        P: 17
      P: 80
        T: 19
  => [50, 20, 80, 15, 17, 19]


### 8.2 Trace mẹo tổng + đi ngược trên một cây tự dựng (đã đối chiếu cách 2 — đối chiếu lại đề gốc)

In [12]:
demo = [[4, 2, 1], [2, 9, 0], [7, 4, 0], [9, 3, 1], [7, 5, 1]]
vals = {v for p, c, _ in demo for v in (p, c)}
print(f"  Σ giá trị phân biệt = {sum(vals)}, Σ con = {sum(c for _, c, _ in demo)} -> gốc = {sum(vals) - sum(c for _, c, _ in demo)}")
parent_of = {c: p for p, c, _ in demo}
v = 3; path = [v]
while v in parent_of:
    v = parent_of[v]; path.append(v)
print("  đi ngược từ 3:", " -> ".join(map(str, path)))
r = build_sum_trick(demo)
draw(r)
assert serialize(r) == serialize(build_dict_set(demo))
print("  =>", serialize(r))

  Σ giá trị phân biệt = 30, Σ con = 23 -> gốc = 7
  đi ngược từ 3: 3 -> 9 -> 2 -> 4 -> 7
    gốc: 7
      T: 5
      P: 4
        T: 2
          P: 9
            T: 3
  => [7, 5, 4, None, None, 2, None, None, 9, 3]


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def createBinaryTree(self, descriptions: List[List[int]]) -> Optional[TreeNode]:
        nodes = {}
        children = set()
        for p, c, is_left in descriptions:
            if p not in nodes:
                nodes[p] = TreeNode(p)
            if c not in nodes:
                nodes[c] = TreeNode(c)
            if is_left:
                nodes[p].left = nodes[c]
            else:
                nodes[p].right = nodes[c]
            children.add(c)
        for v, node in nodes.items():
            if v not in children:            # đỉnh duy nhất không có cha
                return node
        return None

for desc_, exp in examples:
    assert serialize(Solution().createBinaryTree(desc_)) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

1. **Một nút cho mỗi giá trị:** từ điển chỉ tạo `TreeNode(v)` khi `v` chưa có ⇒ mọi mô tả nhắc tới `v` thao tác trên cùng một đối tượng.
2. **Mọi cạnh được nối đúng chiều:** mỗi mô tả gán đúng con trỏ `left`/`right` theo `isLeft`. Cây hợp lệ ⇒ không có hai mô tả cùng gán một phía của cùng một cha (không ghi đè).
3. **Gốc:** trong cây có gốc, mọi đỉnh khác gốc có đúng một cha ⇒ xuất hiện trong `children`; gốc thì không. Mọi giá trị đều là khóa của `nodes` (đều xuất hiện trong ít nhất một mô tả, vì `len(descriptions) ≥ 1` và cây liên thông) ⇒ vòng tìm gốc trả về đúng một đỉnh.
4. **Kết quả là cả cây:** từ gốc, theo các con trỏ đã nối, tới được mọi đỉnh (cây liên thông).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(descriptions)` (số đỉnh = `n + 1`).

| Cách | Time | Space |
|---|---|---|
| 1. Tìm gốc vét cạn | `O(n²)` | `O(n)` |
| 2. Từ điển + tập con | `O(n)` | `O(n)` |
| 3. Bậc vào | `O(n)` | `O(n)` |
| 4. Mẹo tổng | `O(n)` | `O(n)` (từ điển), `O(1)` thêm |
| 5. Đi ngược lên | `O(n + h)` | `O(n)` |

In [14]:
def random_tree_descriptions(n_nodes, rnd, shape="random"):
    vals = rnd.sample(range(1, 10**5 + 1), n_nodes)
    root = vals[0]; slots = [(root, 1), (root, 0)]; desc = []
    for i, v in enumerate(vals[1:], 1):
        if shape == "chain":
            p = vals[i - 1]; side = rnd.randint(0, 1)
        else:
            k = rnd.randrange(len(slots)); p, side = slots[k]; slots[k] = slots[-1]; slots.pop()
            slots += [(v, 1), (v, 0)]
        desc.append([p, v, side])
    rnd.shuffle(desc)
    return desc, root

rnd = random.Random(2196)
for shape in ["random", "chain"]:
    desc_, root_ = random_tree_descriptions(10**4 + 1, rnd, shape)
    row = []
    for name, fn in [("dict_set", build_dict_set), ("indegree", build_indegree), ("sum_trick", build_sum_trick),
                     ("climb", build_climb), ("Solution", Solution().createBinaryTree)]:
        t0 = time.perf_counter(); r = fn(desc_); row.append(f"{name}={(time.perf_counter() - t0) * 1000:.1f}ms")
        assert r.val == root_
    print(f"{shape:<6}: " + ", ".join(row))
small_desc, small_root = random_tree_descriptions(2001, rnd)
t0 = time.perf_counter(); assert build_brute(small_desc).val == small_root
print(f"brute (chỉ 2000 cạnh): {(time.perf_counter() - t0) * 1000:.0f} ms — O(n²), 10⁴ cạnh sẽ chậm hơn ~25 lần")

random: dict_set=9.1ms, indegree=10.6ms, sum_trick=11.7ms, climb=5.9ms, Solution=6.0ms
chain : dict_set=6.9ms, indegree=8.4ms, sum_trick=8.7ms, climb=6.9ms, Solution=7.2ms


brute (chỉ 2000 cạnh): 18 ms — O(n²), 10⁴ cạnh sẽ chậm hơn ~25 lần


## 12. Bộ test

In [15]:
ALL = {
    "brute": build_brute,
    "dict_set": build_dict_set,
    "indegree": build_indegree,
    "sum_trick": build_sum_trick,
    "climb": build_climb,
    "Solution": lambda d: Solution().createBinaryTree(d),
}

def check(descriptions, exp=None, root_val=None, skip=()):
    target = set(map(tuple, descriptions))
    sers = []
    for name, fn in ALL.items():
        if name in skip:
            continue
        r = fn([list(d) for d in descriptions])
        assert descriptions_of(r) == target, name                     # validator: sinh lại đúng mô tả
        if root_val is not None:
            assert r.val == root_val, name
        sers.append(serialize(r))
    assert all(s == sers[0] for s in sers)
    if exp is not None:
        assert sers[0] == exp
    return sers[0]

for desc_, exp in examples:
    check(desc_, exp)
print("Ví dụ đề: OK")

check([[1, 2, 1]], [1, 2])                       # một cạnh, con trái
check([[1, 2, 0]], [1, None, 2])                 # con phải
check([[2, 1, 1], [3, 2, 1]], [3, 2, None, 1])   # mô tả từ dưới lên, gốc ở mô tả cuối
check([[5, 6, 0], [5, 4, 1]], [5, 4, 6])
print("Edge case: OK")

rnd = random.Random(2021)
for _ in range(500):
    d, rt = random_tree_descriptions(rnd.randint(2, 15), rnd, rnd.choice(["random", "chain"]))
    check(d, root_val=rt)
print("500 cây ngẫu nhiên nhỏ (thứ tự mô tả xáo trộn): OK")

for shape in ["random", "chain"]:
    d, rt = random_tree_descriptions(10**4 + 1, rnd, shape)
    check(d, root_val=rt, skip=("brute",))
print("Cây lớn 10⁴ cạnh (ngẫu nhiên và dây xích): OK")

Ví dụ đề: OK
Edge case: OK
500 cây ngẫu nhiên nhỏ (thứ tự mô tả xáo trộn): OK


Cây lớn 10⁴ cạnh (ngẫu nhiên và dây xích): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tạo nút mới mỗi lần gặp giá trị (không dùng từ điển)

In [16]:
def bug_new_nodes(descriptions):
    last = {}
    for p, c, left in descriptions:
        pn = TreeNode(p); cn = TreeNode(c)               # SAI: mỗi lần một đối tượng mới
        if left: pn.left = cn
        else: pn.right = cn
        last[p] = pn
    children = {c for _, c, _ in descriptions}
    root = next(v for v in last if v not in children)
    return last[root]

desc_, exp = examples[0]
print("sai :", serialize(bug_new_nodes(desc_)), "| đúng:", exp)

sai : [50, None, 80] | đúng: [50, 20, 80, 15, 17, 19]


**Vì sao sai:** nút `20` được nối vào `50` là một đối tượng **khác** với nút `20` mang các con `15, 17` ⇒ cây bị đứt. **Sửa:** từ điển `giá trị → nút`.

### Lỗi 2 — Lấy `descriptions[0][0]` làm gốc

In [17]:
def bug_first_parent(descriptions):
    nodes = {}
    for p, c, left in descriptions:
        nodes.setdefault(p, TreeNode(p)); nodes.setdefault(c, TreeNode(c))
        setattr(nodes[p], "left" if left else "right", nodes[c])
    return nodes[descriptions[0][0]]                     # SAI

desc_, exp = examples[0]
print("sai :", serialize(bug_first_parent(desc_)), "| đúng:", exp)

sai : [20, 15, 17] | đúng: [50, 20, 80, 15, 17, 19]


**Vì sao sai:** mô tả có thứ tự tùy ý; mô tả đầu là `[20, 15, 1]` nhưng gốc là `50`. **Sửa:** tìm đỉnh không là con.

### Lỗi 3 — Đảo trái/phải (`isLeft == 0` ⇒ trái)

In [18]:
def bug_swap_sides(descriptions):
    nodes, children = {}, set()
    for p, c, left in descriptions:
        nodes.setdefault(p, TreeNode(p)); nodes.setdefault(c, TreeNode(c))
        setattr(nodes[p], "right" if left else "left", nodes[c])     # SAI: đảo nghĩa
        children.add(c)
    return next(n for v, n in nodes.items() if v not in children)

desc_, exp = examples[1]
print("sai :", serialize(bug_swap_sides(desc_)), "| đúng:", exp)

sai : [1, None, 2, 3, None, None, 4] | đúng: [1, 2, None, None, 3, 4]


**Sửa:** `isLeft = 1` ⇒ con **trái**.

### Lỗi 4 — Dùng `setdefault(v, TreeNode(v))` trong vòng lặp nóng (đúng nhưng tạo đối tượng thừa)

In [19]:
created = {"setdefault": 0, "if-not-in": 0}
class CountingNode(TreeNode):
    counter_key = None
    def __init__(self, val=0, left=None, right=None):
        super().__init__(val, left, right)
        created[CountingNode.counter_key] += 1

d_big, _ = random_tree_descriptions(5001, random.Random(4))
CountingNode.counter_key = "setdefault"; nodes = {}
for p, c, l in d_big:
    nodes.setdefault(p, CountingNode(p)); nodes.setdefault(c, CountingNode(c))
CountingNode.counter_key = "if-not-in"; nodes = {}
for p, c, l in d_big:
    if p not in nodes: nodes[p] = CountingNode(p)
    if c not in nodes: nodes[c] = CountingNode(c)
print("số TreeNode được tạo:", created, "| số đỉnh thật:", len(nodes))

số TreeNode được tạo: {'setdefault': 10000, 'if-not-in': 5001} | số đỉnh thật: 5001


**Vì sao là lỗi:** `setdefault(v, TreeNode(v))` luôn **tạo** `TreeNode(v)` trước khi biết `v` đã có — kết quả đúng nhưng tạo gấp đôi số đối tượng (rác). Với `10⁴` mô tả không đáng kể, nhưng trong bài lớn hơn sẽ tốn thời gian/bộ nhớ. **Sửa:** `if v not in nodes`.

### Lỗi 5 — Tuần tự hóa/kiểm thử bằng DFS **đệ quy** trên cây dây xích

In [20]:
def height_iterative(root):
    best, stack = 0, [(root, 1)]
    while stack:
        node, d = stack.pop()
        if node:
            best = max(best, d); stack.append((node.left, d + 1)); stack.append((node.right, d + 1))
    return best

d_chain, _ = random_tree_descriptions(10**4 + 1, random.Random(5), "chain")
h = height_iterative(Solution().createBinaryTree(d_chain))
print("chiều cao cây dây xích:", h, "| giới hạn đệ quy:", sys.getrecursionlimit(), "->", "đệ quy sẽ RecursionError" if h > sys.getrecursionlimit() else "an toàn")

chiều cao cây dây xích: 10001 | giới hạn đệ quy: 1000 -> đệ quy sẽ RecursionError


**Vì sao là lỗi:** lời giải không đệ quy, nhưng **hàm kiểm thử** (in cây, tính chiều cao, so sánh hai cây) viết đệ quy sẽ vỡ ở cây sâu `10⁴`. Ta không chạy bản đệ quy; mọi hàm tiện ích trong notebook đều dùng stack/hàng đợi. **Sửa:** duyệt lặp.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Dữ liệu cạnh "cha–con" theo thứ tự bất kỳ ⇒ **từ điển giá trị → nút**, tạo lười.
- Gốc = **không có cạnh vào** (tập con, bậc vào, hoặc mẹo tổng/XOR).
- Kiểm thử cây: tự viết `serialize` BFS + validator sinh lại cạnh.
- Python: duyệt cây **lặp** khi cây có thể sâu `10⁴`.
- `setdefault(k, Obj())` luôn tạo `Obj()` — dùng `if k not in d` khi tạo tốn kém.

## 15. Biến thể của bài

1. **Kiểm tra mô tả có tạo thành cây nhị phân hợp lệ không** (một gốc, mỗi đỉnh ≤ 1 cha, mỗi phía ≤ 1 con, liên thông, không chu trình) — bài tập 5.
2. **LeetCode 1361 — Validate Binary Tree Nodes.**
3. **LeetCode 105 / 106 — dựng cây từ preorder/inorder** (khác dạng đầu vào).
4. **Xuất ngược** cây thành `descriptions` (hàm `descriptions_of` ở mục 3.4).
5. **Cây tổng quát** (nhiều con): từ điển `giá trị → list con`, gốc tìm như trên.

In [21]:
def validate_1361(n, leftChild, rightChild):
    """LeetCode 1361: các nút 0..n-1 tạo đúng một cây nhị phân?"""
    indeg = [0] * n
    for ch in leftChild + rightChild:
        if ch != -1:
            indeg[ch] += 1
            if indeg[ch] > 1:
                return False
    roots = [i for i in range(n) if indeg[i] == 0]
    if len(roots) != 1:
        return False
    seen, stack = {roots[0]}, [roots[0]]
    while stack:
        u = stack.pop()
        for ch in (leftChild[u], rightChild[u]):
            if ch != -1:
                if ch in seen: return False
                seen.add(ch); stack.append(ch)
    return len(seen) == n

print("1361:", validate_1361(4, [1, -1, 3, -1], [2, -1, -1, -1]), validate_1361(4, [1, -1, 3, -1], [2, 3, -1, -1]), validate_1361(2, [1, 0], [-1, -1]))

1361: True False False


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn gốc | kiểm từng cha | `O(n²)` | `O(n)` | kiểm thử |
| 2. Từ điển + tập con | cha − con | `O(n)` | `O(n)` | **nộp bài** |
| 3. Bậc vào | `indeg == 0` | `O(n)` | `O(n)` | kèm kiểm hợp lệ |
| 4. Mẹo tổng | `Σ đỉnh − Σ con` | `O(n)` | `O(1)` thêm | gọn bộ nhớ |
| 5. Đi ngược | theo `parent_of` | `O(n + h)` | `O(n)` | nền union-find |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `serialize(Solution().createBinaryTree([[3,1,0],[3,2,1],[1,4,1]]))`?

**Bài 2 (tính tay):** Với ví dụ 2, tính gốc bằng mẹo tổng và bằng mẹo XOR.

**Bài 3 (điền TODO):** Hoàn thiện `level_sums(descriptions)` — dựng cây rồi trả về tổng giá trị từng tầng (BFS).

**Bài 4 (tự viết lại):** Viết lại `createBinaryTree` dùng **mẹo XOR** thay vì tập con.

**Bài 5 (mini challenge):** Viết `is_valid_descriptions(descriptions)` trả `True` nếu mô tả tạo đúng một cây nhị phân hợp lệ (không đỉnh nào 2 cha, không phía nào bị gán 2 lần, đúng một gốc, mọi đỉnh tới được từ gốc). Kiểm trên mô tả sinh ngẫu nhiên đúng và các mô tả bị "phá" có chủ đích.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def level_sums(descriptions):
    # TODO: root = Solution().createBinaryTree(...); BFS theo tầng
    pass

# Bài 4 — TODO
def create_xor(descriptions):
    # TODO: x = XOR mọi đỉnh phân biệt ^ XOR mọi con -> gốc
    pass

# Bài 5 — TODO
def is_valid_descriptions(descriptions):
    # TODO: đếm cha của mỗi con, kiểm phía trái/phải không trùng, đúng 1 gốc, duyệt được hết
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Gốc là 3 (không là con).
- Mức 2: 3: trái 2, phải 1; 1: trái 4.
- Mức 3: `[3, 2, 1, None, None, 4]`.

**Bài 2**
- Mức 1: Đỉnh `{1, 2, 3, 4}`, con `{2, 3, 4}`.
- Mức 2: Tổng: `10 − 9 = 1`.
- Mức 3: XOR: `(1^2^3^4) ^ (2^3^4) = 1`.

**Bài 3**
- Mức 1: BFS theo tầng (`for _ in range(len(q))`).
- Mức 2: Cộng giá trị trong tầng.
- Mức 3: Ví dụ 1: `[50, 100, 51]`.

**Bài 4**
- Mức 1: Duyệt `nodes` để XOR mọi đỉnh.
- Mức 2: XOR thêm mọi `c`.
- Mức 3: Đỉnh xuất hiện 2 lần (một lần trong tập đỉnh, một lần làm con) triệt tiêu.

**Bài 5**
- Mức 1: `parent_count[c] ≤ 1`, `(p, isLeft)` không trùng.
- Mức 2: Đúng một đỉnh có `parent_count = 0`.
- Mức 3: Duyệt lặp từ gốc, đếm số đỉnh thăm = số đỉnh (loại chu trình).

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1
r1 = serialize(Solution().createBinaryTree([[3, 1, 0], [3, 2, 1], [1, 4, 1]]))
print("Bài 1:", r1); assert r1 == [3, 2, 1, None, None, 4]

# Bài 2
d2 = examples[1][0]
vals2 = {v for p, c, _ in d2 for v in (p, c)}
xr = 0
for v in vals2: xr ^= v
for _, c, _ in d2: xr ^= c
print("Bài 2: tổng ->", sum(vals2) - sum(c for _, c, _ in d2), "| XOR ->", xr); assert xr == 1

# Bài 3
def level_sums_sol(descriptions):
    root = Solution().createBinaryTree(descriptions); out = []; q = deque([root])
    while q:
        s = 0
        for _ in range(len(q)):
            node = q.popleft(); s += node.val
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
        out.append(s)
    return out

print("Bài 3:", level_sums_sol(examples[0][0])); assert level_sums_sol(examples[0][0]) == [50, 100, 51]

Bài 1: [3, 2, 1, None, None, 4]
Bài 2: tổng -> 1 | XOR -> 1
Bài 3: [50, 100, 51]


In [24]:
# Bài 4
def create_xor_sol(descriptions):
    nodes = {}; x = 0
    for p, c, left in descriptions:
        if p not in nodes: nodes[p] = TreeNode(p)
        if c not in nodes: nodes[c] = TreeNode(c)
        if left: nodes[p].left = nodes[c]
        else: nodes[p].right = nodes[c]
        x ^= c
    for v in nodes: x ^= v
    return nodes[x]

# Bài 5
def is_valid_descriptions_sol(descriptions):
    parent_count, side_used, vals = {}, set(), set()
    for p, c, left in descriptions:
        if p == c or (p, left) in side_used:
            return False
        side_used.add((p, left))
        parent_count[c] = parent_count.get(c, 0) + 1
        if parent_count[c] > 1:
            return False
        vals.update((p, c))
    roots = [v for v in vals if v not in parent_count]
    if len(roots) != 1:
        return False
    adj = {}
    for p, c, _ in descriptions:
        adj.setdefault(p, []).append(c)
    seen, stack = {roots[0]}, [roots[0]]
    while stack:
        u = stack.pop()
        for w in adj.get(u, []):
            if w in seen: return False
            seen.add(w); stack.append(w)
    return len(seen) == len(vals)

rnd = random.Random(5)
for _ in range(500):
    d, rt = random_tree_descriptions(rnd.randint(2, 12), rnd, rnd.choice(["random", "chain"]))
    assert create_xor_sol(d).val == rt and is_valid_descriptions_sol(d)
    bad = [list(x) for x in d]
    kind = rnd.randrange(3)
    if kind == 0:                                       # thêm một cha thứ hai cho một con
        p, c, l = rnd.choice(bad); other = rnd.choice([v for v in {q for q, _, _ in bad} | {c2 for _, c2, _ in bad} if v not in (p, c)] or [10**6])
        bad.append([other, c, rnd.randint(0, 1)])
    elif kind == 1:                                     # gán trùng phía
        p, c, l = rnd.choice(bad); bad.append([p, 10**6 + 1, l])
    else:                                               # thêm một cây rời (hai gốc)
        bad.append([10**6 + 2, 10**6 + 3, 1])
    assert not is_valid_descriptions_sol(bad), (d, bad)
print("Bài 4, Bài 5 OK; XOR ví dụ 1:", serialize(create_xor_sol(examples[0][0])))

Bài 4, Bài 5 OK; XOR ví dụ 1: [50, 20, 80, 15, 17, 19]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
nodes, children = {}, set()
for p, c, is_left in descriptions:
    if p not in nodes: nodes[p] = TreeNode(p)
    if c not in nodes: nodes[c] = TreeNode(c)
    if is_left: nodes[p].left = nodes[c]
    else:       nodes[p].right = nodes[c]
    children.add(c)
return next(node for v, node in nodes.items() if v not in children)
```

**Checklist:**
- [ ] Một nút cho mỗi giá trị (từ điển)?
- [ ] `isLeft = 1` ⇒ trái?
- [ ] Gốc tìm bằng "không là con", không lấy mô tả đầu?
- [ ] Hàm kiểm thử/in cây duyệt **lặp**?
- [ ] Validator sinh lại đúng tập mô tả?
- [ ] Chữ ký đúng: `def createBinaryTree(self, descriptions: List[List[int]]) -> Optional[TreeNode]`?